# Artemis II trajectory

Anton Antonov  
June 2026  
November 2026


----

## Introduction

This notebook shows how the [JPL Horizons](https://ssd.jpl.nasa.gov/horizons/) published trajectory data Artemis II that can be accessed via the Raku package "WWW::HorizonsEphemerisSystem" and how to make corresponding trajectory plots. 

---

## Setup

In [40]:
use WWW::HorizonsEphemerisSystem;
use Data::TypeSystem;
use Data::Summarizers;
use JavaScript::D3;

In [41]:
%%javascript
require.config({
  paths: {
    d3: "https://d3js.org/d3.v7.min",
    d3_3d: "https://unpkg.com/d3-3d@2.0.2/build/d3-3d"
  }
});

require(["d3", "d3_3d"], function(d3, d3_3d) {
  window.d3 = d3;
  window.d33d = d3_3d || window.d33d || {};
  console.log("d3:", d3.version, "d3-3d exports:", Object.keys(window.d33d));
});

In [42]:
my $background = '#1F1F1F';
my $title-color = 'Ivory';

Ivory

In [43]:
#%js
js-d3-list-line-plot(rand xx 30, :$background)

---

## Artemis II trajectory

[JPL Horizons](https://ssd.jpl.nasa.gov/horizons/) published trajectory data that can be accessed via the Raku package "WWW::HorizonsEphemerisSystem". A user of that package can ask for the state (position and velocity) of the spacecraft. The data assumes a launch on April 1 which is obviously dependent on finding no reason to delay the launch due to technical issues. JPL Horizons refers to the spacecraft using the numerical identifier "-1024".

Date-time spec for the flight of Artemis II:

In [27]:
my @dates = "2026-04-02 01:58:33", "2026-04-10 23:50:00", "10 m";

[2026-04-02 01:58:33 2026-04-10 23:50:00 10 m]

Same date spec by derive via `DateTime` objects:

In [28]:
my $start = DateTime.new(:2026year, :4month, :2day, :1hour, :59minute, :0timezone);
my $end = DateTime.new(:2026year, :4month, :10day, :23hour, :50minute, :0timezone);
my @dates = [$start, $end].map({ "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" });
@dates .= push('10 m');
say (:@dates);

dates => [2026-04-02 01:59:00 2026-04-10 23:50:00 10 m]


Retrieve the data from Horizons System:

In [29]:
my @artemis-position = horizons-ephemeris-data(
    'state',
    ['@-1024', { center => '500@399', :@dates }],
    'position',
    :modifier<data>,
);
deduce-type( @artemis-position)

Vector(Struct([Calendar Date (TDB), Date, Vx, Vy, Vz, X, Y, Z], [Str, Str, Num, Num, Num, Num, Num, Num]), 1284)

Summary of the trajectory positions:

In [30]:
sink records-summary(@artemis-position, field-names => <Date X Y Z>)

+---------------------------+-------------------------------+-------------------------------+-------------------------------+
| Date                      | X                             | Y                             | Z                             |
+---------------------------+-------------------------------+-------------------------------+-------------------------------+
| 2461136.985416667 => 1    | Min    => -133944.7216295758  | Min    => -389915.6166533006  | Min    => -37988.06141233233  |
| 2461139.138194445 => 1    | 1st-Qu => -118689.12058892331 | 1st-Qu => -342385.12213832245 | 1st-Qu => -36185.68829686147  |
| 2461132.936805556 => 1    | Mean   => -80722.23595143012  | Mean   => -239921.44518759692 | Mean   => -26041.201960749357 |
| 2461138.478472222 => 1    | Median => -96831.87098686464  | Median => -267500.7169995487  | Median => -30128.67204218567  |
| 2461132.784027778 => 1    | 3rd-Qu => -44171.99114217423  | 3rd-Qu => -145239.3952153462  | 3rd-Qu => -18109.4710422

3D plot of the trajectory:

In [ ]:
#%js
js-d3-list-line-plot3d(
    @artemis-position.map(*<X Y Z>), 
    :$background, 
    box-ratios => Whatever, 
    :axes, 
    :!boxed
)

----

## Combined plot

Let us combine the trajectory of Artemis II with that of the Moon and also point flight's starting point.

Moon position (with respect to Earth):

In [32]:
my @moon-position = horizons-ephemeris-data(
    'state',
    ['301', { center => '500@399', :@dates }],
    'position',
    :modifier<data>,
);
deduce-type( @moon-position)

Vector(Struct([Calendar Date (TDB), Date, Vx, Vy, Vz, X, Y, Z], [Str, Str, Num, Num, Num, Num, Num, Num]), 1284)

Start of Artemis ||:

In [33]:
my @start-point = [[|@artemis-position.head<X Y Z>, 'start', 'point'],]

[[-24493.42692601773 -14430.08391994964 -1293.485690740577 start point]]

Combine data:

In [34]:
my @data = 
    (0, 0, 0, 'Earth', 'point'),
    |@moon-position.map({ [|$_<X Y Z>, 'Moon', 'line'] }), 
    |@artemis-position.map({ [|$_<X Y Z>, 'Artemis', 'line'] }), 
    |@start-point, 
; 
@data = @data.map({ <x y z group type>.Array Z=> $_.Array })».Hash; 
deduce-type(@data)

Vector((Any), 2570)

In [ ]:
#%js
js-d3-list-point-plot3d(
        @data,
        background => '#1F1F1F',
        height => 800,
        width => 800,
        margins => %(left => 120, bottom => 40),
        title => 'Artemis II trajectory',
        :20title-font-size,
        box-ratios => Whatever,
        :boxed,
        :!axes,
        :!ticks,
        color-scheme => <SteelBlue Silver Pink Red>
);